# Merged Final Dataset Training - 19K Examples

## Dataset: V8.1 + StandUp4AI + YouTube Scraped (Hindi + Bengali)

| Language | Train | Valid | Test |
|----------|-------|-------|------|
| EN | 7,445 | 1,350 | 472 |
| ZH | 2,051 | 446 | 101 |
| HI-LATN | 6,833 | 248 | 252 |
| HI | 38 | 5 | 5 |
| FR | 830 | 107 | 115 |
| ES | 236 | 17 | 33 |
| BN | 1,676 | 200 | 200 |
| Total | 19,109 | 2,989 | 1,790 |

**Target:** F1 >= 0.75 for ALL languages


In [ ]:
!pip install -q torch transformers scikit-learn pandas numpy

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/merged_final'
print(f'Data directory: {DATA_DIR}')

In [ ]:
import os
# Check files exist
print('Files in merged_final:')
for f in os.listdir(DATA_DIR):
    fpath = os.path.join(DATA_DIR, f)
    size = os.path.getsize(fpath) / (1024*1024)
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Load merged dataset from Google Drive
def load_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

print('Loading merged dataset...')
train_df = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_df = load_jsonl(f'{DATA_DIR}/valid.jsonl')
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

print('\nLanguage distribution (train):')
print(train_df['language'].value_counts())

In [ ]:
# Prepare labels
def prepare_labels(df):
    if 'labels' in df.columns and isinstance(df.iloc[0].get('labels', []), list):
        df['ex_label'] = df['labels'].apply(lambda x: 1 if any(v == 1 for v in x) else 0)
    elif 'label' in df.columns:
        df['ex_label'] = df['label']
    return df

train_df = prepare_labels(train_df)
val_df = prepare_labels(val_df)
print(f'Label dist (train): {train_df["ex_label"].value_counts().to_dict()}')

In [ ]:
class MergedDS(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts, self.labels, self.tok, self.max_len = texts, labels, tok, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        txt = ' '.join(self.texts[idx]) if isinstance(self.texts[idx], list) else str(self.texts[idx])
        enc = self.tok(txt, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'label': torch.tensor(self.labels[idx], dtype=torch.long)}

class XLMR(torch.nn.Module):
    def __init__(self, mname):
        super().__init__()
        self.enc = AutoModel.from_pretrained(mname)
        self.drop = torch.nn.Dropout(0.2)
        self.cls = torch.nn.Linear(self.enc.config.hidden_size, 2)
    def forward(self, ids, mask):
        return self.cls(self.drop(self.enc(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]))

In [ ]:
train_texts = train_df['words'].apply(lambda x: ' '.join(x)).tolist()
train_labels = train_df['ex_label'].tolist()
val_texts = val_df['words'].apply(lambda x: ' '.join(x)).tolist()
val_labels = val_df['ex_label'].tolist()
print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = MergedDS(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = MergedDS(val_texts, val_labels, tokenizer, MAX_LEN)
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f'Batches: train={len(train_ld)}, val={len(val_ld)}')

In [ ]:
model = XLMR(MODEL_NAME).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(train_ld)*EPOCHS), num_training_steps=len(train_ld)*EPOCHS)
print('Model ready!')

In [ ]:
def train_ep(m, ld, opt, sch):
    m.train()
    tot, preds, labs = 0, [], []
    for b in ld:
        opt.zero_grad()
        logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
        loss = torch.nn.CrossEntropyLoss()(logits, b['label'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step(); sch.step()
        tot += loss.item()
        preds.extend(torch.argmax(logits, 1).cpu().numpy())
        labs.extend(b['label'].numpy())
    return tot/len(ld), f1_score(labs, preds, average='weighted')

def eval_ep(m, ld):
    m.eval()
    tot, preds, labs = 0, [], []
    with torch.no_grad():
        for b in ld:
            logits = m(b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
            loss = torch.nn.CrossEntropyLoss()(logits, b['label'].to(DEVICE))
            tot += loss.item()
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            labs.extend(b['label'].numpy())
    return tot/len(ld), f1_score(labs, preds, average='weighted'), preds, labs

In [ ]:
best_f1 = 0
hist = {'tl': [], 'tf': [], 'vl': [], 'vf': []}
print('Training on 19K merged dataset...')
for ep in range(EPOCHS):
    tl, tf = train_ep(model, train_ld, opt, sched)
    vl, vf, _, _ = eval_ep(model, val_ld)
    hist['tl'].append(tl); hist['tf'].append(tf); hist['vl'].append(vl); hist['vf'].append(vf)
    print(f'Ep{ep+1}/{EPOCHS}: TL={tl:.4f} TF={tf:.4f} VL={vl:.4f} VF={vf:.4f}')
    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), '/tmp/best_merged.pt')
        print(f'  -> Saved F1={best_f1:.4f}')
print(f'Best Val F1: {best_f1:.4f}')

In [ ]:
model.load_state_dict(torch.load('/tmp/best_merged.pt'))
_, _, preds, actual = eval_ep(model, val_ld)
val_df['pred'] = preds
print('='*50)
print('PER-LANGUAGE RESULTS')
print('='*50)
results = {}
for lang in sorted(val_df['language'].unique()):
    ld = val_df[val_df['language'] == lang]
    if len(ld) > 0:
        f1 = f1_score(ld['ex_label'], ld['pred'], average='weighted')
        results[lang] = f1
        st = 'PASS' if f1 >= 0.75 else 'FAIL'
        print(f'{lang.upper()}: F1={f1:.4f} [{st}] (n={len(ld)})')
overall = f1_score(actual, preds, average='weighted')
st = 'PASS' if overall >= 0.75 else 'FAIL'
print(f'\nOVERALL: F1={overall:.4f} [{st}]')

In [ ]:
# Save to Google Drive
OUTPUT_DIR = '/content/drive/MyDrive/merged_models'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_merged.pt')
model.save_pretrained(f'{OUTPUT_DIR}/xlmr_merged')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/xlmr_merged')
with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump({'overall': overall, 'per_lang': results, 'best': best_f1}, f, indent=2)
print(f'Saved to {OUTPUT_DIR}!')